In [2]:
# ============================================================
# Cell 1: 安装依赖 + 启动 Ollama (GPU) + 构建 RAG
# ============================================================

import subprocess, os, sys

# --- 检测运行环境 ---
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f'📍 运行环境: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "其他"}')

# --- GPU 检测 ---
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU 已连接: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU! 请在 Kaggle 右侧 Settings → Accelerator 选择 GPU')
    print('   或在 Colab: Runtime → Change runtime type → GPU')

# --- 系统依赖 ---
!apt-get update -qq && apt-get install -y -qq zstd ffmpeg > /dev/null 2>&1

# --- Python 依赖 ---
# 用 whisper tiny/base 降低延迟; faster-whisper 可选但更快
!pip install -q openai-whisper edge-tts aiohttp gradio langchain langchain-community faiss-cpu sentence-transformers

# --- 安装 Ollama ---
!curl -fsSL https://ollama.com/install.sh | sh

# --- 启动 Ollama (GPU 模式) ---
import time, threading

def start_ollama():
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    env['OLLAMA_ORIGINS'] = '*'
    # ✅ 关键: 让 Ollama 使用 GPU
    env['OLLAMA_GPU_LAYERS'] = '999'  # 全部层放 GPU
    subprocess.Popen(['ollama', 'serve'], env=env)

threading.Thread(target=start_ollama, daemon=True).start()
print('⏳ 等待 Ollama 启动...')

# 等待 Ollama 就绪 (轮询而非固定 sleep)
import requests
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 (耗时 {i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ Ollama 启动超时，请检查日志')

# --- 下载模型 (选更小/更快的模型以降低延迟) ---
# llama3.1 (8B) 在 T4 上推理较慢，如需更快可换 qwen2:1.5b 或 phi3:mini
print('📥 下载 LLM 模型...')
!ollama pull llama3.1
print('✅ LLM 模型就绪')

# 验证模型是否使用 GPU
!echo '---'; curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; d=json.load(sys.stdin); [print(f'  模型: {m[\"name\"]}') for m in d.get('models',[])]"

# --- RAG 知识库构建 ---
import glob
from pathlib import Path

try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ✅ 自动检测知识库路径 (兼容 Colab / Kaggle)
possible_kb_paths = [
    '/kaggle/input/knowledge-base/RAG/data',      # ← 改这里，匹配你的 dataset 结构
    '/kaggle/input',
    '/content/drive/MyDrive/Colab Notebooks/RAG/data/knowledge_base',
    os.path.join(WORK_DIR, 'knowledge_base'),
]

# Colab: 挂载 Drive
if IS_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
    except:
        pass

def load_hwu_documents(root_dir):
    docs = []
    md_files = glob.glob(os.path.join(root_dir, '**/*.md'), recursive=True)
    print(f'  扫描: {root_dir} → {len(md_files)} 个 .md 文件')
    for fp in md_files:
        try:
            text = Path(fp).read_text(encoding='utf-8')
            lines = text.split('\n', 2)
            if len(lines) < 3:
                continue
            source = lines[0].replace('Source:', '').strip()
            topic = lines[1].replace('Topic:', '').strip()
            content = lines[2].strip()
            docs.append(Document(
                page_content=content,
                metadata={'source': source, 'topic': topic, 'filename': Path(fp).name}
            ))
        except Exception as e:
            print(f'  ⚠️ {fp}: {e}')
    return docs

vector_db = None
for kb_path in possible_kb_paths:
    if os.path.exists(kb_path):
        documents = load_hwu_documents(kb_path)
        if documents:
            chunks = RecursiveCharacterTextSplitter(
                chunk_size=600, chunk_overlap=100
            ).split_documents(documents)
            print(f'  切分为 {len(chunks)} 个片段，构建向量索引...')
            embeddings = HuggingFaceEmbeddings(
                model_name='sentence-transformers/all-MiniLM-L6-v2',
                model_kwargs={'device': 'cuda' if HAS_GPU else 'cpu'}  # ✅ Embedding 也上 GPU
            )
            vector_db = FAISS.from_documents(chunks, embeddings)
            print(f'  ✅ RAG 向量库构建完成 ({len(chunks)} chunks)')
            break

if vector_db is None:
    print('⚠️ 未找到知识库，将以纯 LLM 模式运行')
    print(f'  Kaggle: 请上传 .md 文件到 Dataset')
    print(f'  Colab: 请确认 Google Drive 中有知识库')

print('\n🎉 所有准备工作完成！请运行下一个 Cell 启动语音助手。')

📍 运行环境: Kaggle
✅ GPU 已连接: Tesla T4 (14.6 GB)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%###########################                  78.8%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
⏳ 等待 Ollama 启动...
[GIN] 2026/02/26 - 16:18:32 | 200 |     540.706µs |             ::1 | GET      "/api/tags"
✅ Ollama 启动成功 (耗时 1s)
📥 下载 LLM 模型...
]11;?\

Error: listen tcp 0.0.0.0:11434: bind: address already in use


[GIN] 2026/02/26 - 16:18:37 | 200 |      54.036µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕█████████████

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ RAG 向量库构建完成 (26903 chunks)

🎉 所有准备工作完成！请运行下一个 Cell 启动语音助手。


In [38]:
import subprocess, os, time, requests

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
subprocess.Popen(['ollama', 'serve'])

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 ({i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ 启动失败，查看日志:')
    !cat /tmp/ollama.log 2>/dev/null || echo "无日志"

time=2026-02-26T17:02:08.129Z level=INFO source=routes.go:1663 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[* http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_RE

[GIN] 2026/02/26 - 17:02:09 | 200 |     720.153µs |             ::1 | GET      "/api/tags"
✅ Ollama 启动成功 (2s)


time=2026-02-26T17:02:09.251Z level=INFO source=types.go:42 msg="inference compute" id=GPU-924e6f79-d165-951f-5d48-30d5af126bd6 filter_id="" library=CUDA compute=7.5 name=CUDA1 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:05.0 type=discrete total="15.0 GiB" available="14.5 GiB"
time=2026-02-26T17:02:09.251Z level=INFO source=types.go:42 msg="inference compute" id=GPU-e7f48ea5-a5ed-994b-a529-c9e8cd789d12 filter_id="" library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="15.0 GiB" available="13.6 GiB"
time=2026-02-26T17:02:09.251Z level=INFO source=routes.go:1768 msg="vram-based default context" total_vram="30.0 GiB" default_num_ctx=32768


In [32]:
!pip install -q openai-whisper edge-tts aiohttp gradio langchain langchain-community faiss-cpu sentence-transformers nest_asyncio

In [39]:
# ============================================================
# Cell 2: 实时语音通话 (优化版: 低延迟 + TTS 修复 + GPU 加速)
# ============================================================

import gradio as gr
import json
import requests
import whisper
import edge_tts
import asyncio
import os
import time
import re
import tempfile
import shutil
import numpy as np
import scipy.io.wavfile as wavfile
import torch
import threading
import traceback

# ✅ 关键修复: 允许嵌套事件循环 (Kaggle/Jupyter 自带 loop)
import nest_asyncio
nest_asyncio.apply()

# ============================================================
# Whisper 加载 (GPU)
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Whisper 设备: {device}')

WHISPER_MODEL_SIZE = 'base'
stt_model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)

if device == 'cuda':
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (GPU)')
else:
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (CPU - 会较慢)')

# ============================================================
# VAD 参数
# ============================================================
SILENCE_THRESHOLD = 0.012
SILENCE_DURATION  = 1.2
MIN_SPEECH_DURATION = 0.3

def detect_speech(audio_chunk, sr):
    if audio_chunk is None or len(audio_chunk) == 0:
        return False
    if audio_chunk.dtype == np.int16:
        audio_float = audio_chunk.astype(np.float32) / 32768.0
    elif audio_chunk.dtype == np.int32:
        audio_float = audio_chunk.astype(np.float32) / 2147483648.0
    else:
        audio_float = audio_chunk.astype(np.float32)
    rms = np.sqrt(np.mean(audio_float ** 2))
    return rms > SILENCE_THRESHOLD

def save_buffer_to_wav(audio_buffer, sr):
    if not audio_buffer:
        return None
    combined = np.concatenate(audio_buffer)
    if len(combined) < sr * MIN_SPEECH_DURATION:
        return None
    path = os.path.join(tempfile.gettempdir(), f'voice_{int(time.time()*1000)}.wav')
    if combined.dtype != np.int16:
        if np.max(np.abs(combined)) <= 1.0:
            combined = (combined * 32767).astype(np.int16)
        else:
            combined = combined.astype(np.int16)
    wavfile.write(path, sr, combined)
    return path

# ============================================================
# ASR
# ============================================================
def speech_to_text(audio_path):
    if not audio_path:
        return ''
    try:
        t0 = time.time()
        result = stt_model.transcribe(
            audio_path,
            language='en',
            fp16=(device == 'cuda'),
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            compression_ratio_threshold=2.4,
        )
        text = result['text'].strip()
        elapsed = time.time() - t0

        hallucinations = {
            'thank you', 'thanks for watching', 'subscribe',
            'you', 'bye', 'the end', '字幕', '感谢观看',
            '请不吝点赞', '订阅', '谢谢大家', ''
        }
        if text.lower().strip('.!? ') in hallucinations:
            return ''

        print(f'🎤 识别 ({elapsed:.2f}s): {text}')
        return text
    except Exception as e:
        print(f'❌ ASR Error: {e}')
        return ''

# ============================================================
# TTS (修复版: nest_asyncio + 重试 + 彻底清理 URL)
# ============================================================
def _run_tts_sync(text, voice, output_path, retries=2):
    for attempt in range(retries):
        try:
            loop = asyncio.get_event_loop()
            communicate = edge_tts.Communicate(text, voice, rate='+5%')
            loop.run_until_complete(communicate.save(output_path))

            if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
                return True
            else:
                print(f'  ⚠️ TTS 文件太小 (attempt {attempt+1}), 重试...')
        except Exception as e:
            print(f'  ⚠️ TTS attempt {attempt+1} failed: {e}')
            time.sleep(0.5)
    return False

def text_to_speech(text, voice='en-US-EmmaNeural'):
    if not text or len(text.strip()) < 2:
        return None

    print(f'🔈 TTS 输入: [{text[:100]}]')

    clean = text.split('---')[0]
    clean = re.sub(r'\(https?://[^\)]+\)', '', clean)
    clean = re.sub(r'https?://\S+', '', clean)
    clean = re.sub(r'For more information,?\s*visit:?\s*', '', clean)
    clean = re.sub(r'[*#_>~`\[\](){}|]', '', clean)
    clean = re.sub(r'\\n|\\r', ' ', clean)
    clean = re.sub(r'\n+', '. ', clean)
    clean = re.sub(r'\s{2,}', ' ', clean).strip()
    clean = re.sub(r'\.{2,}', '.', clean)

    if not clean or len(clean) < 2:
        print('⚠️ TTS: 清理后文本为空')
        return None

    if len(clean) > 1500:
        cut = clean[:1500].rfind('.')
        if cut > 100:
            clean = clean[:cut+1]
        else:
            clean = clean[:1500]

    has_cjk = bool(re.search(r'[\u4e00-\u9fff]', clean))
    v = 'zh-CN-XiaoxiaoNeural' if has_cjk else voice

    path = os.path.join(tempfile.gettempdir(), f'tts_{int(time.time()*1000)}.mp3')

    if _run_tts_sync(clean, v, path):
        print(f'🔊 TTS OK: {path} ({len(clean)} chars)')
        return path
    else:
        print('❌ TTS 最终失败')
        return None

# ============================================================
# ✅ 强制 Gradio 每次都重新播放音频
# ============================================================
def make_unique_audio(audio_path):
    if audio_path is None:
        return None
    try:
        unique_path = os.path.join(tempfile.gettempdir(), f'play_{int(time.time()*1000)}.mp3')
        shutil.copy2(audio_path, unique_path)
        return unique_path
    except Exception as e:
        print(f'⚠️ copy error: {e}')
        return audio_path

# ============================================================
# RAG 检索
# ============================================================
def retrieve_context(query):
    if 'vector_db' not in globals() or vector_db is None:
        return ''
    try:
        docs = vector_db.as_retriever(search_kwargs={'k': 3}).invoke(query)
        parts = []
        for i, d in enumerate(docs):
            source_url = d.metadata.get('source', 'N/A')
            topic = d.metadata.get('topic', 'N/A')
            parts.append(f'[Doc {i}] (Source: {source_url} | Topic: {topic}):\n{d.page_content}')
        return '\n\n'.join(parts)
    except Exception as e:
        print(f'RAG Error: {e}')
        return ''

# ============================================================
# LLM 调用 (流式输出 + 原始 system instruction)
# ============================================================
def call_llm(user_text, chat_history):
    ctx = retrieve_context(user_text)
    prompt = f"""
    You are the AI Student Services Assistant for Heriot-Watt University (HWU).

    Here is the retrieved knowledge from the university database:
    [CONTEXT START]
    {ctx}
    [CONTEXT END]

    You must strictly follow these logical steps:

    **Step 1: Check Topic Relevance**
    Is the user's question about Student Services, Campus Life, or University Administration?
    - IF NO (e.g., football, coding, math, general chat):
      REPLY EXACTLY: "I apologize, I can only answer questions regarding Student Services at Heriot-Watt University."
      (STOP HERE).

    **Step 2: Check University Scope**
    Is the question specifically about Heriot-Watt University?
    - IF NO (e.g., Edinburgh University, general UK visa rules not specific to HWU):
      REPLY EXACTLY: "I apologize, I can only answer questions regarding Heriot-Watt University."
      (STOP HERE).

    **Step 3: Check Context Availability**
    Is the answer in the [CONTEXT] above?
    - IF YES: Answer politely using ONLY the context. You MUST include the source URL link from the context at the end of your answer so the user can find more details. Format: "For more information, visit: [URL]"
    - IF NO: REPLY EXACTLY: "I apologize, I do not have this specific information at the moment."

    **Output Guidelines:**
    - If the user asks in Chinese, translate the standard replies into natural Chinese.
    - ALWAYS include the reference source link when answering from context.
    - Keep answers concise and conversational.
    """

    msgs = [{'role': 'system', 'content': prompt}]
    recent_history = (chat_history or [])[-3:]
    for u, a in recent_history:
        msgs.append({'role': 'user', 'content': u})
        if a:
            msgs.append({'role': 'assistant', 'content': a})
    msgs.append({'role': 'user', 'content': user_text})

    try:
        t0 = time.time()
        resp = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': 'llama3.1',
                'messages': msgs,
                'stream': True,
                'options': {
                    'num_predict': 200,
                    'temperature': 0.3,
                    'num_ctx': 2048,
                }
            },
            timeout=60,
            stream=True
        )

        full_text = ''
        for line in resp.iter_lines():
            if line:
                try:
                    chunk = json.loads(line)
                    if 'message' in chunk and 'content' in chunk['message']:
                        full_text += chunk['message']['content']
                    if chunk.get('done', False):
                        break
                except json.JSONDecodeError:
                    continue

        elapsed = time.time() - t0
        print(f'🤖 LLM ({elapsed:.1f}s): {full_text[:100]}...')
        return full_text if full_text else 'Sorry, I could not generate a response.'

    except requests.Timeout:
        return 'Sorry, the response timed out. Please try again.'
    except Exception as e:
        print(f'❌ LLM Error: {e}')
        return f'Sorry, service error: {e}'

# ============================================================
# 核心: 流式音频 VAD 回调
# ✅ 关键修复: 没有新音频时用 gr.update() 而不是 None
#    防止 streaming mic 的持续回调把正在播放的音频覆盖掉
# ============================================================
def process_streaming_audio(audio_chunk, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }

    if state_data.get('processing', False):
        return state_data, state_data.get('chat_history', []), gr.update(), '🔄 AI 正在回复...'

    if audio_chunk is None:
        return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ 等待说话...'

    sr, audio_data = audio_chunk
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    now = time.time()
    is_voice = detect_speech(audio_data, sr)

    if is_voice:
        state_data['audio_buffer'].append(audio_data)
        state_data['is_speaking'] = True
        state_data['speech_detected'] = True
        state_data['silence_start'] = None
        return state_data, state_data.get('chat_history', []), gr.update(), '🗣️ 正在听你说话...'
    else:
        if state_data['speech_detected']:
            state_data['audio_buffer'].append(audio_data)
            if state_data['silence_start'] is None:
                state_data['silence_start'] = now

            silence_elapsed = now - state_data['silence_start']

            if silence_elapsed >= SILENCE_DURATION:
                state_data['processing'] = True
                print(f'\n⏹️ 静音 {silence_elapsed:.1f}s，开始处理...')

                wav_path = save_buffer_to_wav(state_data['audio_buffer'], sr)
                state_data['audio_buffer'] = []
                state_data['is_speaking'] = False
                state_data['speech_detected'] = False
                state_data['silence_start'] = None

                if wav_path is None:
                    state_data['processing'] = False
                    return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ 语音太短，请重试...'

                pipeline_start = time.time()

                user_text = speech_to_text(wav_path)
                if not user_text:
                    state_data['processing'] = False
                    return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ 没听清，请再说一次...'

                ai_text = call_llm(user_text, state_data.get('chat_history', []))
                audio_out = text_to_speech(ai_text)

                total_latency = time.time() - pipeline_start
                print(f'⏱️ 总延迟: {total_latency:.1f}s (ASR + LLM + TTS)')

                history = state_data.get('chat_history', [])
                history.append((user_text, ai_text))
                state_data['chat_history'] = history
                state_data['processing'] = False

                # ✅ 只有这里真正更新音频，用唯一文件名强制重新播放
                return state_data, history, make_unique_audio(audio_out), f'🎙️ 完毕 ({total_latency:.1f}s)，继续说话...'
            else:
                remaining = SILENCE_DURATION - silence_elapsed
                return state_data, state_data.get('chat_history', []), gr.update(), f'🤫 停顿中... ({remaining:.1f}s)'
        else:
            return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ 等待说话...'

# ============================================================
# 文字输入 (备用)
# ============================================================
def handle_text_input(user_text, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }
    if not user_text or not user_text.strip():
        return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ 等待说话...', ''

    ai_text = call_llm(user_text, state_data.get('chat_history', []))
    audio_out = text_to_speech(ai_text)
    history = state_data.get('chat_history', [])
    history.append((user_text, ai_text))
    state_data['chat_history'] = history

    return state_data, history, make_unique_audio(audio_out), '🎙️ AI 回复完毕，继续说话...', ''

# ============================================================
# Gradio 界面
# ============================================================
with gr.Blocks(title='HWU Voice Call') as demo:
    gr.Markdown('# 📞 HWU AI Student Services - Live Voice Call')
    gr.Markdown(
        '**使用方法:** 点击麦克风开始通话，直接说话，'
        f'停顿 {SILENCE_DURATION}s 后 AI 自动回复。\n\n'
        f'**环境:** GPU={torch.cuda.is_available()} | '
        f'Whisper={WHISPER_MODEL_SIZE} | Device={device}'
    )

    call_data = gr.State(value={
        'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
        'speech_detected': False, 'chat_history': [], 'processing': False
    })

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label='对话记录', height=420)
        with gr.Column(scale=1):
            status = gr.Textbox(value='🔴 未开始', label='通话状态', interactive=False)
            audio_player = gr.Audio(label='🔊 AI 回复', autoplay=True, interactive=False)
            gr.Markdown('---')
            mic = gr.Audio(
                sources=['microphone'], streaming=True,
                label='🎤 麦克风 (点击开始通话)'
            )
            clear_btn = gr.Button('🗑️ 清空对话')

    with gr.Row():
        text_input = gr.Textbox(
            label='💬 文字输入 (备用)',
            placeholder='也可以打字提问...',
            scale=4
        )
        send_btn = gr.Button('发送', variant='primary', scale=1)

    mic.stream(
        fn=process_streaming_audio,
        inputs=[mic, call_data],
        outputs=[call_data, chatbot, audio_player, status]
    )
    send_btn.click(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )
    text_input.submit(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )

    def clear_all():
        return (
            {'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
             'speech_detected': False, 'chat_history': [], 'processing': False},
            [], None, '🎙️ 已清空'
        )
    clear_btn.click(fn=clear_all, outputs=[call_data, chatbot, audio_player, status])

print('\n🚀 启动实时语音通话...')
demo.launch(debug=True, share=True)

🖥️ Whisper 设备: cuda
✅ Whisper base 已加载 (GPU)

🚀 启动实时语音通话...


/tmp/ipykernel_1061/1522497388.py:401: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='对话记录', height=420)
/tmp/ipykernel_1061/1522497388.py:401: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label='对话记录', height=420)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://c2c1bb4412b694ba06.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



⏹️ 静音 1.5s，开始处理...
🎤 识别 (0.22s): Can you hear me?


time=2026-02-26T17:02:26.359Z level=INFO source=server.go:431 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 36805"
llama_model_loader: loaded meta data with 29 key-value pairs and 292 tensors from /root/.ollama/models/blobs/sha256-667b0c1932bc6ffc593ed1d03f895bf2dc8dc6df21db3042284a6f4416b06a29 (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - k

[GIN] 2026/02/26 - 17:02:31 | 200 |  5.467965028s |             ::1 | POST     "/api/chat"
🤖 LLM (5.5s): I apologize, I can only answer questions regarding Student Services at Heriot-Watt University....
🔈 TTS 输入: [I apologize, I can only answer questions regarding Student Services at Heriot-Watt University.]
🔊 TTS OK: /tmp/tts_1772125351633.mp3 (94 chars)
⏱️ 总延迟: 6.1s (ASR + LLM + TTS)

⏹️ 静音 1.6s，开始处理...
🎤 识别 (0.24s): Can you tell me something about accommodation?
[GIN] 2026/02/26 - 17:02:50 | 200 |  4.194879248s |             ::1 | POST     "/api/chat"
🤖 LLM (4.2s): When choosing your accommodation, it's essential to consider how you'll feel sharing facilities like...
🔈 TTS 输入: [When choosing your accommodation, it's essential to consider how you'll feel sharing facilities like]
🔊 TTS OK: /tmp/tts_1772125370443.mp3 (304 chars)
⏱️ 总延迟: 9.8s (ASR + LLM + TTS)

⏹️ 静音 1.5s，开始处理...
🎤 识别 (0.22s): What's the price of it?
[GIN] 2026/02/26 - 17:03:29 | 200 |  3.228561643s |             ::1 | 

🔊 TTS OK: /tmp/tts_1772125727607.mp3 (124 chars)
⏱️ 总延迟: 4.2s (ASR + LLM + TTS)
